In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score, ConfusionMatrixDisplay
from itertools import combinations
from tqdm.notebook import tqdm
from pathlib import Path
import warnings

# model

In [2]:
def combine_categories(df: pd.DataFrame, combination_map: dict, target_col: str = 'FL_UDSD') -> pd.DataFrame:
    """
    Combine multiple categories in the target column into single categories.
    
    Args:
        df: Input dataframe
        target_col: Column containing categories to combine
        combination_map: Dictionary where keys are new category names and values are lists of 
                        categories to combine into that new category.
                        Example: {'SCD/Impaired': ['Subjective Cognitive Decline', 'Impaired Not SCD/MCI'],
                                 'Normal/SCD': ['Normal cognition', 'Subjective Cognitive Decline']}
    
    Returns:
        pd.DataFrame: DataFrame with combined categories
    """
    df = df.copy()
    
    # Apply each combination
    for new_category, old_categories in combination_map.items():
        df[target_col] = df[target_col].replace(old_categories, new_category)
    
    return df

In [3]:
df = pd.read_csv('../data/clinical/clinical_preprocessed.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2134 entries, 0 to 2133
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   PTID                2134 non-null   int64  
 1   VISITYR             2134 non-null   int64  
 2   CDRSUM              2134 non-null   float64
 3   CDRGLOB             2134 non-null   float64
 4   MMSE                2019 non-null   float64
 5   HVLT_DR             1849 non-null   float64
 6   LASSI_A_CR2         1830 non-null   float64
 7   LASSI_B_CR1         1826 non-null   float64
 8   LASSI_B_CR2         1810 non-null   float64
 9   APOE4S              1903 non-null   float64
 10  PTAU_217_CONCNTRTN  672 non-null    float64
 11  AMYLPET             1430 non-null   float64
 12  FL_UDSD             2120 non-null   float64
 13  NACCETPR            1611 non-null   float64
dtypes: float64(12), int64(2)
memory usage: 233.5 KB


In [5]:
diagnosis_order = ['Normal cognition', 'Subjective Cognitive Decline', 'Impaired Not SCD/MCI',
                   'Early MCI', 'Late MCI', 'Dementia']
diagnosis_map = {i+1: label for i, label in enumerate(diagnosis_order)}

diagnosis_map

{1: 'Normal cognition',
 2: 'Subjective Cognitive Decline',
 3: 'Impaired Not SCD/MCI',
 4: 'Early MCI',
 5: 'Late MCI',
 6: 'Dementia'}

In [6]:
# df['FL_UDSD_CAT'] = df['FL_UDSD'].map(diagnosis_map)
# df['FL_UDSD_CAT']

## Removing some columns
These columns are for informative purposes ['CDRGLOB','AMYLPET', 'NACCETPR'],

In [7]:
REMOVE_FEATURES = ['CDRGLOB','AMYLPET', 'NACCETPR']
df_filter = df.drop(columns=REMOVE_FEATURES).dropna()
df_filter.info()

<class 'pandas.core.frame.DataFrame'>
Index: 591 entries, 6 to 2072
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   PTID                591 non-null    int64  
 1   VISITYR             591 non-null    int64  
 2   CDRSUM              591 non-null    float64
 3   MMSE                591 non-null    float64
 4   HVLT_DR             591 non-null    float64
 5   LASSI_A_CR2         591 non-null    float64
 6   LASSI_B_CR1         591 non-null    float64
 7   LASSI_B_CR2         591 non-null    float64
 8   APOE4S              591 non-null    float64
 9   PTAU_217_CONCNTRTN  591 non-null    float64
 10  FL_UDSD             591 non-null    float64
dtypes: float64(9), int64(2)
memory usage: 55.4 KB


In [9]:
df_filter['FL_UDSD'].value_counts()

FL_UDSD
4.0    218
3.0    144
1.0     79
5.0     65
6.0     58
2.0     27
Name: count, dtype: int64

In [ ]:
def exhaustive_feature_search(
    X, y, groups, cv, estimator,
    min_features=2, max_features=None,
    scoring=('f1_macro', 'balanced_accuracy'),
    primary_metric='f1_macro',
    n_jobs=-1,
):
    """Evaluate every feature combination with size in [min_features, max_features].

    Iterates over all subsets of the columns of X whose cardinality falls
    within [min_features, max_features] and scores each one with grouped
    cross-validation. Useful for small feature sets where brute-force search
    is tractable; cost grows as the sum of binomial coefficients over the
    requested range, so mind the combinatorial explosion.

    Note on scores: uses sklearn.model_selection.cross_validate, which
    reports held-out TEST scores per fold (not training scores). Each
    combination is scored on every fold's test split and aggregated to
    mean/std.

    Parameters
    ----------
    X : pandas.DataFrame
        Feature matrix. Column names are used to enumerate subsets.
    y : array-like of shape (n_samples,)
        Target vector aligned with X.
    groups : array-like of shape (n_samples,)
        Group labels passed to cv so samples from the same group stay on
        the same side of each split (e.g. patient IDs to prevent leakage).
    cv : cross-validation splitter
        Any splitter accepting groups (e.g. StratifiedGroupKFold).
    estimator : sklearn ensemble model
    min_features : int, default=2
        Smallest subset size to evaluate (inclusive).
    max_features : int or None, default=None
        Largest subset size to evaluate (inclusive). None means all
        columns of X.
    scoring : tuple/list/dict of str, default=('f1_macro', 'balanced_accuracy')
        Any scoring strings accepted by sklearn.model_selection.cross_validate.
    primary_metric : str, default='f1_macro'
        Metric used to sort the results. Must appear in `scoring`.
    n_jobs : int, default=-1
        Parallelism passed to cross_validate. -1 uses all cores.

    Returns
    -------
    pandas.DataFrame
        One row per subset, sorted by mean_{primary_metric} descending, with
        columns: n_features, features (list of column names), and
        mean_{m} / std_{m} pairs for every metric m in `scoring`.
    """
    all_features = X.columns.tolist()
    if max_features is None:
        max_features = len(all_features)

    metrics = list(scoring) if not isinstance(scoring, dict) else list(scoring.keys())
    if primary_metric not in metrics:
        raise ValueError(f"primary_metric={primary_metric!r} not in scoring={metrics}")

    results = []
    for k in range(min_features, max_features + 1):
        combos = list(combinations(all_features, k))
        print(f"Evaluating {len(combos)} combinations of size {k}...")
        for feats in tqdm(combos, leave=False):
            feats = list(feats)
            cv_res = cross_validate(
                estimator, X[feats], y,
                groups=groups, cv=cv, scoring=scoring, n_jobs=n_jobs,
                return_train_score=False,
            )
            row = {'n_features': k, 'features': feats}
            for m in metrics:
                fold_scores = cv_res[f'test_{m}']
                row[f'mean_{m}'] = fold_scores.mean()
                row[f'std_{m}']  = fold_scores.std()
            results.append(row)

    return (
        pd.DataFrame(results)
          .sort_values(f'mean_{primary_metric}', ascending=False)
          .reset_index(drop=True)
    )


In [ ]:
X = df_filter.drop(columns=['PTID', "VISITYR", 'FL_UDSD'])
y = df_filter["FL_UDSD"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(max_iter=3000, random_state=42)

In [ ]:
X.info()

In [ ]:
y.value_counts()

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
    combo_results = exhaustive_feature_search(
        X, y,
        groups=df_filter['PTID'],
        cv=cv,
        estimator=model,
        min_features=2,
        max_features=len(X.columns),
    )


In [ ]:
combo_results.to_csv("results/lr_results_6class.csv", index=False)

## Combine SCD + Impaired Not SCD/MCI
Merge classes 2 (Subjective Cognitive Decline) and 3 (Impaired Not SCD/MCI) into a single class and re-run the exhaustive feature search.

In [ ]:
SCD_CODE = 2
IMPAIRED_NOT_SCD_CODE = 3
PRE_MCI_CODE = 2

df_combined = df_filter.copy()

df_combined = combine_categories(
    df_combined,
    combination_map={PRE_MCI_CODE: [SCD_CODE, IMPAIRED_NOT_SCD_CODE]},
    target_col='FL_UDSD',
)

# Renumber so codes are contiguous: 1=Normal, 2=Pre-MCI, 3=EMCI, 4=LMCI, 5=Dementia
renumber_map = {1: 1, 2: 2, 4: 3, 5: 4, 6: 5}
df_combined['FL_UDSD'] = df_combined['FL_UDSD'].map(renumber_map)

df_combined['FL_UDSD'].value_counts().sort_index()

In [ ]:
diagnosis_order = ['Normal cognition', 'Pre-MCI', 
                   'Early MCI', 'Late MCI', 'Dementia']
diagnosis_map = {i+1: label for i, label in enumerate(diagnosis_order)}

diagnosis_map

In [ ]:
X_combined = df_combined.drop(columns=['PTID', 'VISITYR', 'FL_UDSD'])
y_combined = df_combined['FL_UDSD']

cv_combined = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
model_combined = LogisticRegression(max_iter=3000, random_state=42)

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
    combo_results_combined = exhaustive_feature_search(
        X_combined, y_combined,
        groups=df_combined['PTID'],
        cv=cv_combined,
        estimator=model_combined,
        min_features=2,
        max_features=len(X_combined.columns),
    )

In [ ]:
combo_results_combined.to_csv("results/lr_results_SCD_Imp.csv", index=False)